In [1]:
import os
import itertools
import multiprocessing as mp

import numpy as np
import pandas as pd
import supervision as sv

from concurrent.futures import ProcessPoolExecutor
from collections import defaultdict

from trackers import OCSORTTracker
from trackers.eval import evaluate_mot_sequences


# Shared detection utilities (same logic as in the SORT/ByteTrack SoccerNet notebooks)

def build_dets_index(det_list):
    dets_by_frame = defaultdict(list)
    for line in det_list:
        frame_id = int(line.split(",")[0])
        dets_by_frame[frame_id].append(line)
    return dets_by_frame


def get_detections_from_dict(frame_id, dets_by_frame):
    dets = []
    for line in dets_by_frame[frame_id]:
        det = line.split(",")  # "frame_id,-1,left,top,width,height,conf,-1,-1,-1"
        x1 = int(det[2])
        y1 = int(det[3])
        x2 = int(det[4]) + int(det[2])
        y2 = int(det[5]) + int(det[3])
        conf = 1.0
        dets.append([x1, y1, x2, y2, conf])
    return dets

In [2]:
def run_ocsort_soccernet_with_params(
    lost_track_buffer: int,
    minimum_iou_threshold: float,
    minimum_consecutive_frames: int,
    direction_consistency_weight: float,
    high_conf_det_threshold: float,
    delta_t: int,
    eval_set: str = "test",
):
    """Run OC-SORT on all SoccerNet sequences for a given hyperparameter set.

    Uses split-specific detections/GT for `train` and the official
    `SoccerNet_tracking_2022_all_dets`/`SoccerNet_tracking_2022_all_gts` for `test`.
    """

    tracker = OCSORTTracker(
        high_conf_det_threshold=high_conf_det_threshold,
        minimum_iou_threshold=minimum_iou_threshold,
        minimum_consecutive_frames=minimum_consecutive_frames,
        delta_t=delta_t,
        direction_consistency_weight=direction_consistency_weight,
        lost_track_buffer=lost_track_buffer,
    )

    outputs_root = "OCSORT_outputs_soccernet"
    os.makedirs(outputs_root, exist_ok=True)
    save_dir = os.path.join(
        outputs_root,
        f"{eval_set}_L{lost_track_buffer}_I{minimum_iou_threshold}_"
        f"C{minimum_consecutive_frames}_D{direction_consistency_weight}_"
        f"H{high_conf_det_threshold}_T{delta_t}",
    )
    os.makedirs(save_dir, exist_ok=True)

    # Choose detections and GT depending on split
    if eval_set == "train":
        det_root = "SoccerNet_dets/SoccerNet_tracking/train"
        gt_root = "TrackEval/data/gt/SoccerNet_tracking/train"
    elif eval_set == "test":
        det_root = "SoccerNet_dets/SoccerNet_tracking_2022_all_dets"
        gt_root = "TrackEval/data/gt/SoccerNet_tracking/SoccerNet_tracking_2022_all_gts"
    else:
        raise ValueError(f"Unsupported eval_set: {eval_set}")

    for seq in sorted(os.listdir(det_root)):
        tracker.reset()
        seq_name = seq.split("__")[0]

        with open(os.path.join(det_root, seq), "r") as f_det:
            det_list = f_det.readlines()
            dets_by_frame = build_dets_index(det_list)

        last_frame = int(det_list[-1].split(",")[0])
        output_lines = []

        for frame_id in range(1, last_frame + 1):
            raw_dets = get_detections_from_dict(frame_id, dets_by_frame)
            if raw_dets:
                raw_dets = np.array(raw_dets)
                dets = sv.Detections(
                    xyxy=raw_dets[:, :4],
                    confidence=raw_dets[:, 4],
                )
            else:
                dets = sv.Detections.empty()

            dets = tracker.update(detections=dets)

            for tid, (left, top, right, bottom) in zip(dets.tracker_id, dets.xyxy):
                if tid == -1:
                    continue

                width = right - left
                height = bottom - top

                output_lines.append(
                    f"{frame_id},{int(tid)},{round(left,1)},{round(top,1)},{round(width,1)},{round(height,1)},-1,-1,-1,-1\n"
                )

        save_path = os.path.join(save_dir, seq_name + ".txt")
        with open(save_path, "w") as f:
            f.writelines(output_lines)

    result = evaluate_mot_sequences(
        gt_dir=gt_root,
        tracker_dir=save_dir,
        metrics=["CLEAR", "HOTA", "Identity"],
    )

    MOTA = result.to_dict()["aggregate"]["CLEAR"]["MOTA"]
    HOTA = result.to_dict()["aggregate"]["HOTA"]["HOTA"]
    IDF1 = result.to_dict()["aggregate"]["Identity"]["IDF1"]

    print(
        f"OCSORT | split:{eval_set} | L:{lost_track_buffer}, I:{minimum_iou_threshold}, "
        f"C:{minimum_consecutive_frames}, D:{direction_consistency_weight}, "
        f"H:{high_conf_det_threshold}, T:{delta_t} -> "
        f"HOTA: {HOTA:.3f}, IDF1: {IDF1:.3f}, MOTA: {MOTA:.3f}"
    )

    return {
        "lost_track_buffer": lost_track_buffer,
        "minimum_iou_threshold": minimum_iou_threshold,
        "minimum_consecutive_frames": minimum_consecutive_frames,
        "direction_consistency_weight": direction_consistency_weight,
        "high_conf_det_threshold": high_conf_det_threshold,
        "delta_t": delta_t,
        "HOTA": HOTA,
        "IDF1": IDF1,
        "MOTA": MOTA,
    }

In [3]:
# Define OC-SORT hyperparameter search space for SoccerNet (same as MOT17/SportsMOT)

lost_track_buffer_candidates = [10, 30, 60]
minimum_iou_threshold_candidates = [0.1, 0.3, 0.5]
minimum_consecutive_frames_candidates = [3, 5]
direction_consistency_weight_candidates = [0.0, 0.2, 0.5]
high_conf_det_threshold_candidates = [0.4, 0.6, 0.8]
delta_t_candidates = [1, 3]

all_candidate_lists = [
    lost_track_buffer_candidates,
    minimum_iou_threshold_candidates,
    minimum_consecutive_frames_candidates,
    direction_consistency_weight_candidates,
    high_conf_det_threshold_candidates,
    delta_t_candidates,
]

parameter_combinations = list(itertools.product(*all_candidate_lists))
print(f"Total OC-SORT parameter combinations for SoccerNet: {len(parameter_combinations)}")

Total OC-SORT parameter combinations for SoccerNet: 324


In [4]:
def run_ocsort_tuning_parallel(parameter_combinations, eval_set: str = "train"):
    num_workers = os.cpu_count()
    print(f"Running {len(parameter_combinations)} OC-SORT combinations on {eval_set} with {num_workers} workers")

    ctx = mp.get_context("fork")
    all_results = []

    with ProcessPoolExecutor(max_workers=num_workers, mp_context=ctx) as executor:
        futures = []
        for i, comb in enumerate(parameter_combinations):
            print(f"Submitting combination {i + 1}/{len(parameter_combinations)}")
            (
                lost_track_buffer,
                minimum_iou_threshold,
                minimum_consecutive_frames,
                direction_consistency_weight,
                high_conf_det_threshold,
                delta_t,
            ) = comb
            futures.append(
                executor.submit(
                    run_ocsort_soccernet_with_params,
                    lost_track_buffer,
                    minimum_iou_threshold,
                    minimum_consecutive_frames,
                    direction_consistency_weight,
                    high_conf_det_threshold,
                    delta_t,
                    eval_set,
                )
            )

        for i, f in enumerate(futures):
            try:
                result = f.result()
                all_results.append(result)
                print(f"[{i + 1}/{len(parameter_combinations)}] combination finished.")
            except Exception as e:
                print(f"[{i + 1}/{len(parameter_combinations)}] combination FAILED: {e}")

    df = pd.DataFrame(all_results)
    print("OC-SORT hyperparameter tuning complete. Results stored in 'ocsort_soccernet_tuning_df'.")
    print(df.head())

    df.to_csv("ocsort_soccernet_tuning.csv", index=False)
    return df


ocsort_soccernet_tuning_df = run_ocsort_tuning_parallel(parameter_combinations, eval_set="train")

best_idx = ocsort_soccernet_tuning_df["HOTA"].idxmax()
best_row = ocsort_soccernet_tuning_df.loc[best_idx]
print("\nBest OC-SORT HOTA row (train):\n", best_row)

best_params = dict(
    lost_track_buffer=int(best_row["lost_track_buffer"]),
    minimum_iou_threshold=float(best_row["minimum_iou_threshold"]),
    minimum_consecutive_frames=int(best_row["minimum_consecutive_frames"]),
    direction_consistency_weight=float(best_row["direction_consistency_weight"]),
    high_conf_det_threshold=float(best_row["high_conf_det_threshold"]),
    delta_t=int(best_row["delta_t"]),
)
print("Best params:", best_params)

Running 324 OC-SORT combinations on train with 10 workers
Submitting combination 1/324
Submitting combination 2/324
Submitting combination 3/324
Submitting combination 4/324
Submitting combination 5/324
Submitting combination 6/324
Submitting combination 7/324
Submitting combination 8/324
Submitting combination 9/324
Submitting combination 10/324
Submitting combination 11/324
Submitting combination 12/324
Submitting combination 13/324
Submitting combination 14/324
Submitting combination 15/324
Submitting combination 16/324
Submitting combination 17/324
Submitting combination 18/324
Submitting combination 19/324
Submitting combination 20/324
Submitting combination 21/324
Submitting combination 22/324
Submitting combination 23/324
Submitting combination 24/324
Submitting combination 25/324
Submitting combination 26/324
Submitting combination 27/324
Submitting combination 28/324
Submitting combination 29/324
Submitting combination 30/324
Submitting combination 31/324
Submitting combinatio

OCSORT | split:train | L:10, I:0.1, C:3, D:0.0, H:0.6, T:1 -> HOTA: 0.844, IDF1: 0.792, MOTA: 0.976
OCSORT | split:train | L:10, I:0.1, C:3, D:0.2, H:0.6, T:3 -> HOTA: 0.845, IDF1: 0.794, MOTA: 0.977
OCSORT | split:train | L:10, I:0.1, C:3, D:0.2, H:0.6, T:1 -> HOTA: 0.849, IDF1: 0.799, MOTA: 0.977OCSORT | split:train | L:10, I:0.1, C:3, D:0.0, H:0.4, T:3 -> HOTA: 0.844, IDF1: 0.792, MOTA: 0.976

OCSORT | split:train | L:10, I:0.1, C:3, D:0.0, H:0.4, T:1 -> HOTA: 0.844, IDF1: 0.792, MOTA: 0.976
[1/324] combination finished.
[2/324] combination finished.
[3/324] combination finished.
OCSORT | split:train | L:10, I:0.1, C:3, D:0.0, H:0.6, T:3 -> HOTA: 0.844, IDF1: 0.792, MOTA: 0.976
OCSORT | split:train | L:10, I:0.1, C:3, D:0.0, H:0.8, T:3 -> HOTA: 0.844, IDF1: 0.792, MOTA: 0.976
OCSORT | split:train | L:10, I:0.1, C:3, D:0.0, H:0.8, T:1 -> HOTA: 0.844, IDF1: 0.792, MOTA: 0.976
[4/324] combination finished.
[5/324] combination finished.
[6/324] combination finished.
OCSORT | split:train

In [5]:
# Final test run + submission zip using best_params from train tuning

import shutil

if "best_params" not in globals() or best_params is None:
    raise RuntimeError("best_params is not defined. Run the tuning cell first.")

print("Running OC-SORT on SoccerNet test with:", best_params)
test_result = run_ocsort_soccernet_with_params(
    lost_track_buffer=best_params["lost_track_buffer"],
    minimum_iou_threshold=best_params["minimum_iou_threshold"],
    minimum_consecutive_frames=best_params["minimum_consecutive_frames"],
    direction_consistency_weight=best_params["direction_consistency_weight"],
    high_conf_det_threshold=best_params["high_conf_det_threshold"],
    delta_t=best_params["delta_t"],
    eval_set="test",
)


Running OC-SORT on SoccerNet test with: {'lost_track_buffer': 60, 'minimum_iou_threshold': 0.1, 'minimum_consecutive_frames': 3, 'direction_consistency_weight': 0.2, 'high_conf_det_threshold': 0.4, 'delta_t': 1}


OCSORT | split:test | L:60, I:0.1, C:3, D:0.2, H:0.4, T:1 -> HOTA: 0.829, IDF1: 0.779, MOTA: 0.968
